# Customer Churn Prediction — Exploratory Data Analysis

This notebook walks through the full data exploration process.
It complements the production pipeline code in `app/` with visual insights.

**Why separate notebooks from production code:**
- Notebooks are for exploration and communication
- Production code is modular, tested, and version-controlled
- Never put business logic only in notebooks — always extract to .py files

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from config import RAW_DATA_PATH
from app.data.download_data import get_or_create_dataset
from app.data.preprocess import preprocess_dataframe
from app.data.feature_engineering import engineer_features

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

## 1. Load Raw Data

In [ ]:
df_raw = get_or_create_dataset(RAW_DATA_PATH)
print(f'Shape: {df_raw.shape}')
print(f'Churn rate: {(df_raw["Churn"] == "Yes").mean():.1%}')
df_raw.head()

In [ ]:
# Check data types and missing values
print('Data types and null counts:')
df_raw.info()

## 2. Target Distribution — Class Imbalance Check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Churn distribution
churn_counts = df_raw['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#28a745', '#dc3545'], startangle=90)
axes[0].set_title('Churn Distribution (Class Imbalance)', fontsize=14)

# Churn by contract
contract_churn = df_raw.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean()
).sort_values(ascending=False)
contract_churn.plot(kind='bar', ax=axes[1], color='#1f77b4', edgecolor='white')
axes[1].set_title('Churn Rate by Contract Type', fontsize=14)
axes[1].set_ylabel('Churn Rate')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print('KEY INSIGHT: Month-to-month customers churn 3-4x more than two-year contracts')
print('This makes contract_risk our most important engineered feature')

## 3. Tenure Analysis — The "New Customer" Effect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure distribution by churn
churned = df_raw[df_raw['Churn'] == 'Yes']['tenure']
stayed  = df_raw[df_raw['Churn'] == 'No']['tenure']

axes[0].hist(churned, bins=30, alpha=0.7, color='#dc3545', label='Churned')
axes[0].hist(stayed,  bins=30, alpha=0.7, color='#28a745', label='Stayed')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')
axes[0].set_title('Tenure Distribution by Churn Status')
axes[0].legend()

# Monthly charges vs tenure
churn_num = (df_raw['Churn'] == 'Yes').astype(int)
df_raw_temp = df_raw.copy()
df_raw_temp['TotalCharges'] = pd.to_numeric(df_raw_temp['TotalCharges'], errors='coerce').fillna(0)

axes[1].scatter(
    df_raw_temp['tenure'], df_raw_temp['MonthlyCharges'],
    c=churn_num, cmap='RdYlGn_r', alpha=0.3, s=10
)
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Monthly Charges ($)')
axes[1].set_title('Monthly Charges vs Tenure\n(Red=Churned, Green=Stayed)')

plt.tight_layout()
plt.show()

print(f'Mean tenure (churned): {churned.mean():.1f} months')
print(f'Mean tenure (stayed):  {stayed.mean():.1f} months')
print('\nKEY INSIGHT: New customers (low tenure) are most likely to churn')
print('This justifies our tenure_bucket feature engineering')

## 4. Feature Engineering Preview

In [ ]:
# Run the full pipeline
df_clean    = preprocess_dataframe(df_raw)
df_featured = engineer_features(df_clean)

print(f'Original shape:  {df_raw.shape}')
print(f'After preprocess: {df_clean.shape}')
print(f'After feature eng: {df_featured.shape}')
print(f'\nNew features added: {df_featured.shape[1] - df_raw.shape[1] + 1}')

In [ ]:
# Correlation heatmap of key features with churn
numeric_cols = df_featured.select_dtypes(include=[np.number]).columns.tolist()
corr_with_churn = df_featured[numeric_cols].corr()['Churn'].drop('Churn').sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#dc3545' if x > 0 else '#28a745' for x in corr_with_churn]
corr_with_churn.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Feature Correlation with Churn', fontsize=14)
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 5. Run Training

You can run the full training pipeline from this notebook for exploration.

In [ ]:
# Uncomment to run training (takes ~2 minutes)
# from app.training.train import run_training
# metrics = run_training(tune=False)
# print('\nAll model metrics:')
# for model, m in metrics.items():
#     print(f'{model}: ROC-AUC={m["roc_auc"]}, F1={m["f1"]}')